# 1. 할인 패턴 — 언제, 어디서, 얼마나 싸지나

**찾을 현상**
- 출시 6~12개월 사이에 첫 30% 할인이 온다
- 할인이 여름·연말 세일에 몰린다
- 특정 판매처가 Steam보다 먼저, 더 깊게 할인한다

**인사이트**: 게임별로 “지금 사면 손해인지”, “다음 할인이 언제 올지”를 예측 → 구매 타이밍 추천의 근거 (B: timing_rules.json)

**데이터**: ITAD 가격 이력(USD→원화), 다나와 스냅샷

> 지금은 `mock` 데이터(가설을 심은 가짜 값)라 결과는 결론이 아닙니다. 정제 테이블이 채워지면 `load_dataset("db")`로 바꾸세요.

In [ ]:
import pandas as pd
from IPython.display import Image, display

from gamenuri_analysis import a01_discount_pattern as m
from gamenuri_analysis.loader import load_dataset
from gamenuri_analysis.report import fig_dir

In [ ]:
data = load_dataset("mock")  # 실데이터: load_dataset("db") (MariaDB Replica, 읽기 전용)
data.source

## 계산 · 그래프
`run()`이 계산하고 `analysis/out/figures/<분석>/`에 PNG와 summary.json을 저장합니다.

In [ ]:
res = m.run(data)
for t in res.takeaways:
    print('-', t)

In [ ]:
for f in res.figures:
    print(f['title'], '—', f['caption'])
    display(Image(filename=str(fig_dir(m.KEY) / f['file'])))

In [ ]:
for t in res.tables:
    print(t['title'])
    display(pd.DataFrame(t['rows'], columns=t['columns']))

## 직접 들여다보기
`res.dfs`에 중간 결과가 들어 있어요. 아래 셀을 고쳐 가며 탐색해 보세요.

In [ ]:
print(list(res.dfs))
res.dfs['first30'].groupby('tier').months.describe()

## 서비스로 넘기는 것
- 게임별 태그(`res.per_game`) → B: `out/api/game_insights.json` → 게임 상세 화면
- 구매 타이밍 규칙(`res.api`) → B: `out/api/timing_rules.json` → Backend


In [ ]:
print(sum(len(v) for v in res.per_game.values()), '개 태그')
res.api